<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase28_Standards_Gap_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 28
## Standards and required-field gap resolution (synthetic only)
**Google Drive → Google Drive** · Follows **Phase 27 REPAIRED**, not the earlier failed Phase 27.

**Inputs**: Phase 27 repaired shareable JSON + pinned target fields CSV + 48-row required-gap CSV; the exact Phase 26 synthetic FHIR bundle when available, and (optionally) Phase 27 synthetic staging outputs. All files are discovered under **My Drive** and results are saved beside the repaired Phase 27 shareable JSON.

### What this phase actually does
1. Verify Phase 27 provenance and independently fetch/pin the **OMOP CDM v5.5** official field specification for `PERSON`, `PROCEDURE_OCCURRENCE`, and `MEASUREMENT`.
2. Recheck the previously pinned ImageWG field-level CSV without changing its commit.
3. Repair **synthetic-only** invalid/missing DICOM study and series UIDs using deterministic UUID-derived `2.25.*` UIDs **without overwriting the original Phase 26 bundle**. This cannot attest actual ADNI or DICOM file identifiers.
4. Explicitly reconcile which of Phase 27's 48 gaps can be closed synthetically and which require *real source procedure linkage*, *vocabulary validation*, or additional FHIR evidence.
5. Create an **Athena/vocabulary binding review template**; never guess concept IDs, clinical procedures, demographics, or event-field concepts.

**NOT performed:** pyOMOP execution, a target DB load, official HL7 profile validation, an approved Athena vocabulary map, real MRI ETL, source/target cohort equivalence or a four-arm benchmark. Arm C remains nonportable.

**Standards consulted**: [OHDSI CDM v5.5](https://github.com/OHDSI/CommonDataModel), [pinned ImageWG](https://github.com/OHDSI/ImageWG), [DICOM UID rules](https://dicom.nema.org/medical/dicom/current/output/chtml/part05/chapter_9.html), [FHIR R4 ImagingStudy example](https://hl7.org/fhir/R4/imagingstudy-example.xml.html).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import os, json, csv, hashlib, io, re, subprocess, uuid, copy
from datetime import datetime, timezone
from collections import Counter
import urllib.request
import pandas as pd

# Optional: if MyDrive automatic discovery fails, enter the exact folder containing Phase 27 REPAIRED outputs.
PHASE27_FOLDER_OVERRIDE = ""
MYDRIVE = Path('/content/drive/MyDrive')


def digest(data): return hashlib.sha256(data).hexdigest()
def filehash(path): return digest(Path(path).read_bytes())

def safe_find(root, name, exact=True):
    """Avoid Google Drive stale FUSE directory entries that fail Path.stat()."""
    found=[]
    root=Path(root)
    if root.is_dir():
        for directory, _, filenames in os.walk(root, onerror=lambda err: print('Skipped:',err)):
            for candidate in filenames:
                match=(candidate == name) if exact else (candidate.startswith(name) and candidate.endswith('.json'))
                if not match: continue
                p=Path(directory)/candidate
                try:
                    if p.is_file():
                        p.open('rb').close()
                        found.append(p)
                except (FileNotFoundError,PermissionError,OSError):
                    pass
    return sorted(found, key=lambda p: (p.name != name, len(str(p))))

if PHASE27_FOLDER_OVERRIDE.strip():
    wd=Path(PHASE27_FOLDER_OVERRIDE.strip())
    choices=safe_find(wd,'ICHI2027_Phase27_REPAIRED_SHAREABLE.json')
else:
    choices=safe_find(MYDRIVE,'ICHI2027_Phase27_REPAIRED_SHAREABLE.json')
if not choices:
    raise FileNotFoundError('Cannot find Phase27 REPAIRED shareable JSON. Set PHASE27_FOLDER_OVERRIDE above to its Drive folder.')

# Select a candidate with an actual verified repaired payload, not merely a matching name.
P27=None; WORKDIR=None
for p in choices:
    try:
        obj=json.loads(p.read_text(encoding='utf-8'))
        if obj.get('phase')=='27_repaired_pinned_schema_and_synthetic_armA_staging':
            P27=obj; WORKDIR=p.parent; P27_PATH=p; break
    except (OSError,ValueError): continue
if P27 is None: raise RuntimeError('Found Phase27 filename(s), but none contain the repaired Phase27 payload.')

REQ_NAMES={
 'fields':'ICHI2027_Phase27_REPAIRED_PINNED_TARGET_FIELDS.csv',
 'gaps':'ICHI2027_Phase27_REPAIRED_REQUIRED_FIELD_GAPS.csv',
}
INP={k:WORKDIR / name for k,name in REQ_NAMES.items()}
missing=[str(p) for p in INP.values() if not p.is_file()]
if missing: raise FileNotFoundError('Put the matching Phase27 REPAIRED CSVs beside the shareable JSON: '+str(missing))

fields=pd.read_csv(INP['fields'],keep_default_na=False)
gaps=pd.read_csv(INP['gaps'],keep_default_na=False)
print('Verified Drive folder:',WORKDIR)
print('Phase27:',P27_PATH.name,'fields:',len(fields),'gaps:',len(gaps))

Verified Drive folder: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
Phase27: ICHI2027_Phase27_REPAIRED_SHAREABLE.json fields: 23 gaps: 48


In [3]:
# Exact repaired Phase 27 gate and row-level reconciliation.
assert P27['phase']=='27_repaired_pinned_schema_and_synthetic_armA_staging'
assert P27['pinned_source_dual_hash_verified'] is True
assert P27['schema']=={'image_occurrence_fields':11,'image_feature_fields':12,'verified':True}
assert P27['gates']['exact_MICDM_field_level_schema_verified'] is True
assert P27['gates']['synthetic_armA_staging_created'] is True
assert P27['gates']['synthetic_armA_faithful_target_mapping_completed'] is False
assert P27['arm_registry']['C']['executed'] is False
assert {'table','field','required','data_type','source_relpath'}.issubset(fields.columns)
assert {'table','staging_row_id','required_field','reason'}.issubset(gaps.columns)
assert len(fields)==23 and len(gaps)==P27['required_field_gap_count']==48
assert fields.groupby('table').size().to_dict()=={'image_feature':12,'image_occurrence':11}
expected={(x['table'],x['field']):x['count'] for x in P27['required_field_gap_summary']}
actual=gaps.groupby(['table','required_field']).size().to_dict()
assert actual==expected, f'Phase27 gap counts do not match the JSON: {actual}'
assert not gaps.duplicated(['table','staging_row_id','required_field']).any()
assert P27['repaired_from_phase27_imagewg_commit']=='8f0a19bb94e941bb335a097bc9851fab672621c7'
print('PASS: Phase27 provenance, 23 exact fields and 48 required gaps reconciled.')

PASS: Phase27 provenance, 23 exact fields and 48 required gaps reconciled.


## 1. Independently pin official schema definitions

The notebook pins the exact CDM repository HEAD observed during this run, then downloads `OMOP_CDMv5.5_Field_Level.csv` **from that immutable commit**, recording its SHA-256 and field metadata. If network access is unavailable it records an unverified gate, not an invented schema. It also re-fetches the *same* ImageWG pinned commit and checks **raw byte and normalized-text** hashes recorded by Phase 27.

In [4]:
IMAGEWG_COMMIT=P27['repaired_from_phase27_imagewg_commit']
IMAGING_REL='inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
IMAGEWG_URL=f'https://raw.githubusercontent.com/OHDSI/ImageWG/{IMAGEWG_COMMIT}/{IMAGING_REL}'
CORE_REPO='https://github.com/OHDSI/CommonDataModel.git'
CORE_REL='inst/csv/OMOP_CDMv5.5_Field_Level.csv'
CORE_COMMIT_OVERRIDE=''  # Optional immutable reviewed CommonDataModel commit.


def getbytes(url):
    with urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':'ICHI2027-synthetic-audit/1.0'}),timeout=30) as resp:
        return resp.read()

def norm_text_sha(raw):
    text=raw.decode('utf-8-sig').replace('\r\n','\n').replace('\r','\n')
    return digest(text.encode('utf-8'))

imagewg_rechecked=False; imagewg_error=None
try:
    raw=getbytes(IMAGEWG_URL)
    raw_ok=digest(raw)==P27['pinned_field_level_source_sha256']
    text_ok=norm_text_sha(raw)==P27['pinned_field_level_inventory_text_sha256']
    if not (raw_ok or text_ok):
        raise RuntimeError('The pinned ImageWG bytes AND normalized text disagree with Phase27 hashes. STOP.')
    imagewg_rechecked=True
except Exception as exc:
    imagewg_error=str(exc)
    print('ImageWG network recheck not completed:',imagewg_error)

core_verified=False; core_error=None; CORE_COMMIT=None
core_fields=pd.DataFrame(columns=['table','field','required','data_type','fk_table','fk_field','fk_domain'])
try:
    if CORE_COMMIT_OVERRIDE.strip():
        CORE_COMMIT=CORE_COMMIT_OVERRIDE.strip()
    else:
        result=subprocess.run(['git','ls-remote',CORE_REPO,'HEAD'],capture_output=True,text=True,timeout=35,check=True)
        CORE_COMMIT=result.stdout.split()[0]
    assert bool(re.fullmatch('[a-f0-9]{40}',CORE_COMMIT)), 'Not a full immutable commit'
    core_raw=getbytes(f'https://raw.githubusercontent.com/OHDSI/CommonDataModel/{CORE_COMMIT}/{CORE_REL}')
    cdf=pd.read_csv(io.BytesIO(core_raw),dtype=str,keep_default_na=False)
    needed={'cdmTableName','cdmFieldName','isRequired','cdmDatatype'}
    assert needed.issubset(cdf.columns),f'Official schema CSV has unexpected headers: {cdf.columns}'
    subs=cdf[cdf['cdmTableName'].str.lower().isin(['person','procedure_occurrence','measurement'])].copy()
    assert subs.groupby('cdmTableName').size().ge(10).all()
    assert set(subs['cdmTableName'].str.lower())=={'person','procedure_occurrence','measurement'}
    core_fields=pd.DataFrame({
        'table':subs['cdmTableName'].str.lower(),
        'field':subs['cdmFieldName'].str.lower(),
        'required':subs['isRequired'].str.lower().eq('true'),
        'data_type':subs['cdmDatatype'],
        'fk_table':subs.get('fkTableName',pd.Series(['']*len(subs),index=subs.index)),
        'fk_field':subs.get('fkFieldName',pd.Series(['']*len(subs),index=subs.index)),
        'fk_domain':subs.get('fkDomain',pd.Series(['']*len(subs),index=subs.index))
    })
    core_sha=digest(core_raw)
    core_verified=True
except Exception as exc:
    core_error=str(exc)
    print('OMOP v5.5 independent schema pin not completed:',core_error)

print('Pinned ImageWG CSV rechecked:',imagewg_rechecked)
print('Independent OMOP v5.5 core schema verified:',core_verified)
print('Core commit:',CORE_COMMIT if core_verified else '(unverified)')
if core_verified:
    display(core_fields.groupby('table')['required'].agg(fields='size',required='sum'))

Pinned ImageWG CSV rechecked: True
Independent OMOP v5.5 core schema verified: True
Core commit: 4a910305b2cb74a4fc2b2c34baf44eb0542ff03f


,fields,required
table,,
measurement,24,5
person,18,5
procedure_occurrence,16,5


## 2. Repair synthetic DICOM UIDs **only when the exact Phase 26 bundle exists**

The Phase 26 bundle originally used demonstration strings rather than attestable DICOM identifiers. UUID-derived `2.25.*` UIDs are allowed in the DICOM specification. Generate deterministic, distinct synthetic study and series UIDs and add a conventional FHIR R4 ImagingStudy identifier (`system=urn:dicom:uid`, `value=urn:oid:<UID>`). Keep original Phase 26 data untouched. This tests encoding only; it is **not** evidence that real scans have these UIDs.

In [5]:
# Match the exact Phase26 bundle digest recorded in BOTH Phase 26 and Phase 27.
PHASE26_BUNDLE_OVERRIDE=''
expected_bundle_hash=P27['phase26_bundle_sha256']

def find_bundle():
    candidates=[]
    if PHASE26_BUNDLE_OVERRIDE.strip(): candidates=[Path(PHASE26_BUNDLE_OVERRIDE.strip())]
    else:
        candidates=safe_find(WORKDIR,'ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json')
        if not candidates:
            candidates=safe_find(MYDRIVE,'ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json')
    for p in candidates:
        try:
            if filehash(p)==expected_bundle_hash:
                return p
        except (OSError,ValueError): pass
    return None

BUNDLE_PATH=find_bundle()
if BUNDLE_PATH:
    bundle=json.loads(BUNDLE_PATH.read_text(encoding='utf-8'))
    assert bundle.get('resourceType')=='Bundle' and bundle.get('type')=='collection'
    assert filehash(BUNDLE_PATH)==expected_bundle_hash
    print('Exact Phase26 synthetic bundle verified:',BUNDLE_PATH)
else:
    bundle=None
    print('Exact Phase26 bundle not found. UID mapping will remain blocked; other audits still run.')

OCC_STAGE=WORKDIR/'ICHI2027_Phase27_ARM_A_IMAGE_OCCURRENCE_STAGING.csv'
occ_stage=pd.read_csv(OCC_STAGE,keep_default_na=False) if OCC_STAGE.is_file() else None
if occ_stage is None:
    print('Optional exact Phase27 IMAGE_OCCURRENCE staging CSV not found; row-level UID gap closure stays unverified.')

Exact Phase26 synthetic bundle verified: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase26_SYNTHETIC_FHIR_R4_BUNDLE.json


In [6]:
# Pure deterministic UID functions (UUID v5 derived from synthetic source key).
DICOM_UID_RE=re.compile(r'(?:0|[1-9][0-9]*)(?:\.(?:0|[1-9][0-9]*))+\Z')

def valid_dicom_uid(value):
    return isinstance(value,str) and 1<=len(value)<=64 and bool(DICOM_UID_RE.fullmatch(value))

def synthetic_uid(*parts):
    key='|'.join(['ICHI2027_PHASE28_SYNTHETIC_UID',expected_bundle_hash]+[str(p) for p in parts])
    out='2.25.'+str(uuid.uuid5(uuid.NAMESPACE_URL,key).int)
    assert valid_dicom_uid(out)
    return out

assert valid_dicom_uid('2.25.1234567')
assert not valid_dicom_uid('1.2.840.10008.ICHISYN.1.1')
assert synthetic_uid('study','syn-one') != synthetic_uid('series','syn-one',0)

patched_bundle=None; uid_rows=[]; uid_row_gate=False
if bundle is not None:
    patched_bundle=copy.deepcopy(bundle)
    studies=sorted([e['resource'] for e in patched_bundle.get('entry',[]) if e.get('resource',{}).get('resourceType')=='ImagingStudy'],key=lambda r:r['id'])
    assert len(studies)==4, 'Expected exactly four Phase26 synthetic imaging studies'
    for seq,s in enumerate(studies,1):
        sid=s['id']
        # The prior demonstration's single-series assumption is verified here, not applied to general MRI data.
        assert len(s.get('series',[]))==1, ('A multiple-series source requires separate occurrence rows',sid)
        source_series=s['series'][0].get('uid')
        study_uid=synthetic_uid('study',sid)
        series_uid=synthetic_uid('series',sid,0)
        s.setdefault('identifier',[]).append({'system':'urn:dicom:uid','value':'urn:oid:'+study_uid})
        s['series'][0]['uid']=series_uid
        uid_rows.append({
          'study_id':sid,'staging_image_occurrence_id':seq,
          'synthetic_study_uid':study_uid,'synthetic_series_uid':series_uid,
          'replaced_invalid_synthetic_series_uid':bool(not valid_dicom_uid(source_series)),
          'source_bundle_sha256':expected_bundle_hash,
          'status':'SYNTHETIC_UIDS_GENERATED_NO_REAL_DICOM_ATTESTATION'
        })
    assert len({x['synthetic_study_uid'] for x in uid_rows})==4
    assert len({x['synthetic_series_uid'] for x in uid_rows})==4
    assert {x['synthetic_study_uid'] for x in uid_rows}.isdisjoint({x['synthetic_series_uid'] for x in uid_rows})
    assert all(valid_dicom_uid(x['synthetic_study_uid']) and valid_dicom_uid(x['synthetic_series_uid']) for x in uid_rows)
    # Reconcile old staging with sorted studies and synthetic patient ordering.
    if occ_stage is not None and len(occ_stage)==4 and {'image_occurrence_id','person_id','image_occurrence_date'}.issubset(occ_stage.columns):
        persons=sorted([e['resource']['id'] for e in patched_bundle['entry'] if e['resource'].get('resourceType')=='Patient'])
        pids={pid:i for i,pid in enumerate(persons,1)}
        expected=[(seq,pids[s['subject']['reference'].split('/')[-1]],s['started'][:10]) for seq,s in enumerate(studies,1)]
        actual=[(int(r['image_occurrence_id']),int(r['person_id']),str(r['image_occurrence_date'])[:10]) for _,r in occ_stage.sort_values('image_occurrence_id').iterrows()]
        uid_row_gate=(actual==expected)
        if not uid_row_gate: print('Staging row IDs/dates/person IDs DO NOT match the Phase26 bundle; no row-level closure.')
    else:
        print('Staging not available: UID file is a synthetic example; row-level gap closure not asserted.')

uid_manifest=pd.DataFrame(uid_rows,columns=['study_id','staging_image_occurrence_id','synthetic_study_uid','synthetic_series_uid','replaced_invalid_synthetic_series_uid','source_bundle_sha256','status'])
print('Synthetic study/series pairs:',len(uid_manifest),'stage-row reconciliation:',uid_row_gate)

Synthetic study/series pairs: 4 stage-row reconciliation: True


## 3. Disposition all **48** Phase 27 gaps and inventory additional core prerequisites

Only the eight UID gap instances can be marked `SYNTHETIC_FIXTURE_RESOLVED` when the source bundle, staging row IDs and dates reconcile. The 40 other MI-CDM required-field instances **stay blocked** pending linked procedure source evidence or concept review. Synthetic UID repair does **not** make a real imaging cohort, valid vocabulary mapping, or official MI-CDM load.

In [7]:
resolved_uid_keys=set()
if uid_row_gate:
    for r in uid_rows:
        rid=int(r['staging_image_occurrence_id'])
        resolved_uid_keys.update({('image_occurrence',rid,'image_study_uid'),('image_occurrence',rid,'image_series_uid')})

disposition=gaps.copy()
disposition['resolution_status']=[
    'SYNTHETIC_FIXTURE_RESOLVED' if (r['table'],int(r['staging_row_id']),r['required_field']) in resolved_uid_keys
    else 'BLOCKED_SOURCE_PROCEDURE' if r['required_field']=='procedure_occurrence_id'
    else 'BLOCKED_VOCABULARY_VALIDATION' if str(r['required_field']).endswith('_concept_id')
    else 'BLOCKED_VERIFY_SOURCE'
    for _,r in gaps.iterrows()
]
disposition['resolution_evidence']=[
    '2.25 UUIDv5 synthetic UID; exact Phase26 bundle/staging reconciled; real DICOM unattested' if s=='SYNTHETIC_FIXTURE_RESOLVED'
    else 'FHIR ImagingStudy has no source-linked Procedure record; do not synthesize clinical history' if s=='BLOCKED_SOURCE_PROCEDURE'
    else 'No approved Athena concept in the supplied source; do not invent concept_id' if s=='BLOCKED_VOCABULARY_VALIDATION'
    else 'Source not verified'
    for s in disposition['resolution_status']
]
assert len(disposition)==48
resolved=int(disposition['resolution_status'].eq('SYNTHETIC_FIXTURE_RESOLVED').sum())
remaining=int(disposition['resolution_status'].str.startswith('BLOCKED').sum())
assert resolved+remaining==48
assert resolved in (0,8)
print('Phase27 required-field instances: 48; synthetically resolved:',resolved,'still blocked:',remaining)
display(disposition.groupby(['table','required_field','resolution_status']).size().reset_index(name='n'))

Phase27 required-field instances: 48; synthetically resolved: 8 still blocked: 40


,table,required_field,resolution_status,n
0,image_feature,image_feature_concept_id,BLOCKED_VOCABULARY_VALIDATION,16
1,image_feature,image_feature_type_concept_id,BLOCKED_VOCABULARY_VALIDATION,16
2,image_occurrence,image_series_uid,SYNTHETIC_FIXTURE_RESOLVED,4
3,image_occurrence,image_study_uid,SYNTHETIC_FIXTURE_RESOLVED,4
4,image_occurrence,modality_concept_id,BLOCKED_VOCABULARY_VALIDATION,4
5,image_occurrence,procedure_occurrence_id,BLOCKED_SOURCE_PROCEDURE,4


In [8]:
# The official OMOP core schema has further prerequisites beyond the 48 *MI-CDM* gaps.
# Record them without making up person demographics, procedures or vocabulary concepts.
core_reqs=[]
source_assessment={
 'person':{
   'person_id':'SYNTHETIC_ID_AVAILABLE',
   'gender_concept_id':'BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC',
   'race_concept_id':'BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC',
   'ethnicity_concept_id':'BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC',
   'year_of_birth':'BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC'
 },
 'procedure_occurrence':{
   'procedure_occurrence_id':'BLOCKED_NO_LINKED_SOURCE_PROCEDURE',
   'person_id':'SYNTHETIC_ID_AVAILABLE',
   'procedure_date':'IMAGE_DATE_AVAILABLE_NOT_SOURCE_PROCEDURE_ATTESTATION',
   'procedure_concept_id':'BLOCKED_NO_VALIDATED_STANDARD_CONCEPT',
   'procedure_type_concept_id':'BLOCKED_NO_VALIDATED_STANDARD_CONCEPT'
 },
 'measurement':{
   'measurement_id':'SYNTHETIC_ID_AVAILABLE',
   'person_id':'SYNTHETIC_ID_AVAILABLE',
   'measurement_date':'SYNTHETIC_OBSERVATION_DATE_AVAILABLE',
   'measurement_concept_id':'BLOCKED_NO_VALIDATED_STANDARD_CONCEPT',
   'measurement_type_concept_id':'BLOCKED_NO_VALIDATED_STANDARD_CONCEPT'
 }
}
if core_verified:
    for _,r in core_fields[core_fields.required].iterrows():
        table,field=r['table'],r['field']
        core_reqs.append({
           'table':table,'field':field,'required':True,'data_type':r['data_type'],
           'status':source_assessment.get(table,{}).get(field,'REQUIRES_INDIVIDUAL_REVIEW'),
           'source':'OFFICIAL_OMOP_V55_COMMIT_'+CORE_COMMIT
        })
else:
    for table, entries in source_assessment.items():
        for field,status in entries.items():
            core_reqs.append({'table':table,'field':field,'required':'UNVERIFIED','data_type':'',
                              'status':'OFFICIAL_SCHEMA_NOT_VERIFIED; '+status,'source':'UNVERIFIED'})
core_requirements=pd.DataFrame(core_reqs)
# Relational semantics: even though these are optional fields in the imaging extension,
# a feature that links to a MEASUREMENT needs a validated event-field concept.
event_field_block={
  'table':'image_feature', 'field':'image_feature_event_field_concept_id', 'required':'CONDITIONAL_LINK_SEMANTICS',
  'data_type':'integer','status':'BLOCKED_EVENT_FIELD_CONCEPT_NOT_VALIDATED',
  'source':'PINNED_IMAGEWG_REVISION_'+IMAGEWG_COMMIT
}
core_requirements=pd.concat([core_requirements,pd.DataFrame([event_field_block])],ignore_index=True)
display(core_requirements)

,table,field,required,data_type,status,source
0,person,person_id,True,integer,SYNTHETIC_ID_AVAILABLE,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
1,person,gender_concept_id,True,integer,BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
2,person,year_of_birth,True,integer,BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
3,person,race_concept_id,True,integer,BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
4,person,ethnicity_concept_id,True,integer,BLOCKED_UNRECORDED_SYNTHETIC_DEMOGRAPHIC,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
5,procedure_occurrence,procedure_occurrence_id,True,integer,BLOCKED_NO_LINKED_SOURCE_PROCEDURE,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
6,procedure_occurrence,person_id,True,integer,SYNTHETIC_ID_AVAILABLE,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
7,procedure_occurrence,procedure_concept_id,True,integer,BLOCKED_NO_VALIDATED_STANDARD_CONCEPT,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
8,procedure_occurrence,procedure_date,True,date,IMAGE_DATE_AVAILABLE_NOT_SOURCE_PROCEDURE_ATTE...,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...
9,procedure_occurrence,procedure_type_concept_id,True,integer,BLOCKED_NO_VALIDATED_STANDARD_CONCEPT,OFFICIAL_OMOP_V55_COMMIT_4a910305b2cb74a4fc2b2...


## 4. Manual concept and procedure binding review (no auto-filled concept IDs)

This produces a **fillable template**, not a vocabulary approval. Before executing a standards-compliant Arm A: confirm source evidence for a real/synthetic **Procedure** (do not infer one from the imaging study alone); locally review Athena concepts and their domain, `standard_concept`, invalid status and vocabulary version; resolve **MEASUREMENT** values and `IMAGE_FEATURE` event-field semantics. For authorized real ADNI data, existing acquisition identifiers must be attested, never replaced by synthetic UIDs.

In [9]:
# One row per distinct binding question; no invented concept IDs or approvals.
binding_spec=[
 ('image_occurrence','modality_concept_id','ImagingStudy.series.modality.code','MR','MR imaging modality','VALIDATED_STANDARD_CONCEPT'),
 ('image_feature','image_feature_concept_id','Observation.code + Observation.bodySite','hippocampus/entorhinal; LEFT/RIGHT; volume','region-and-side feature meaning','VALIDATED_STANDARD_CONCEPT'),
 ('image_feature','image_feature_type_concept_id','Device + Provenance','synthetic segmentation algorithm output','derived algorithm provenance','VALIDATED_STANDARD_CONCEPT'),
 ('image_feature','image_feature_event_field_concept_id','MEASUREMENT.measurement_id','MEASUREMENT','event-row ID link field','VALIDATED_EVENT_FIELD_CONCEPT'),
 ('measurement','measurement_concept_id','Observation.code + Observation.bodySite','hippocampus/entorhinal; LEFT/RIGHT; volume','numeric volume concept','VALIDATED_STANDARD_CONCEPT'),
 ('measurement','measurement_type_concept_id','Observation/Provenance','synthetic algorithm output','source-type concept','VALIDATED_STANDARD_CONCEPT'),
 ('measurement','unit_concept_id','Observation.valueQuantity','mm3','UCUM unit mapping','VALIDATED_STANDARD_CONCEPT_OR_DOCUMENTED_POLICY'),
 ('procedure_occurrence','procedure_concept_id','source linked Procedure / imaging event','NOT_PRESENT_IN_PHASE26','no clinical event may be invented','LINKED_PROCEDURE_AND_VALIDATED_STANDARD_CONCEPT'),
 ('procedure_occurrence','procedure_type_concept_id','linked Procedure provenance','NOT_PRESENT_IN_PHASE26','source type provenance','LINKED_PROCEDURE_AND_VALIDATED_STANDARD_CONCEPT'),
 ('person','gender_concept_id','synthetic source patient demographics','NOT_PRESENT','target PERSON required field','SOURCE_EVIDENCE_OR_DOCUMENTED_UNKNOWN_POLICY'),
 ('person','race_concept_id','synthetic source patient demographics','NOT_PRESENT','target PERSON required field','SOURCE_EVIDENCE_OR_DOCUMENTED_UNKNOWN_POLICY'),
 ('person','ethnicity_concept_id','synthetic source patient demographics','NOT_PRESENT','target PERSON required field','SOURCE_EVIDENCE_OR_DOCUMENTED_UNKNOWN_POLICY'),
 ('person','year_of_birth','synthetic source patient demographics','NOT_PRESENT','target PERSON required field','SOURCE_EVIDENCE_REQUIRED'),
]
bindings=pd.DataFrame(binding_spec,columns=['table','field','source_evidence','source_code_or_status','review_question','approval_gate'])
for col in ['proposed_concept_id','proposed_vocabulary_id','expected_domain','athena_snapshot_version','reviewer','review_date','approval_status','evidence_reference']:
    bindings[col]=''
assert bindings['proposed_concept_id'].eq('').all()
print('Prepared',len(bindings),'manual binding questions. No concept IDs invented.')
display(bindings[['table','field','source_code_or_status','approval_gate']])

Prepared 13 manual binding questions. No concept IDs invented.


,table,field,source_code_or_status,approval_gate
0,image_occurrence,modality_concept_id,MR,VALIDATED_STANDARD_CONCEPT
1,image_feature,image_feature_concept_id,hippocampus/entorhinal; LEFT/RIGHT; volume,VALIDATED_STANDARD_CONCEPT
2,image_feature,image_feature_type_concept_id,synthetic segmentation algorithm output,VALIDATED_STANDARD_CONCEPT
3,image_feature,image_feature_event_field_concept_id,MEASUREMENT,VALIDATED_EVENT_FIELD_CONCEPT
4,measurement,measurement_concept_id,hippocampus/entorhinal; LEFT/RIGHT; volume,VALIDATED_STANDARD_CONCEPT
5,measurement,measurement_type_concept_id,synthetic algorithm output,VALIDATED_STANDARD_CONCEPT
6,measurement,unit_concept_id,mm3,VALIDATED_STANDARD_CONCEPT_OR_DOCUMENTED_POLICY
7,procedure_occurrence,procedure_concept_id,NOT_PRESENT_IN_PHASE26,LINKED_PROCEDURE_AND_VALIDATED_STANDARD_CONCEPT
8,procedure_occurrence,procedure_type_concept_id,NOT_PRESENT_IN_PHASE26,LINKED_PROCEDURE_AND_VALIDATED_STANDARD_CONCEPT
9,person,gender_concept_id,NOT_PRESENT,SOURCE_EVIDENCE_OR_DOCUMENTED_UNKNOWN_POLICY


## 5. Save Drive outputs and read back

Even when network checks or optional source files are unavailable, this notebook writes an honest status report to Drive. The phase **cannot** claim faithful Arm A completion solely because its synthetic UID test passes. The original Phase 26/27 files are never overwritten.

In [10]:
OUT={
 'shareable':WORKDIR/'ICHI2027_Phase28_SHAREABLE.json',
 'gap_disposition':WORKDIR/'ICHI2027_Phase28_GAP_DISPOSITION.csv',
 'uid_manifest':WORKDIR/'ICHI2027_Phase28_SYNTHETIC_UID_MANIFEST.csv',
 'core_audit':WORKDIR/'ICHI2027_Phase28_PINNED_OMOP55_CORE_FIELDS.csv',
 'core_requirements':WORKDIR/'ICHI2027_Phase28_CORE_REQUIREMENTS.csv',
 'bindings':WORKDIR/'ICHI2027_Phase28_CONCEPT_AND_PROCEDURE_BINDING_TEMPLATE.csv',
 'patched_fhir':WORKDIR/'ICHI2027_Phase28_SYNTHETIC_FHIR_UID_PATCHED.json'
}

# Write CSVs, preserving a header even for an unavailable optional source.
disposition.to_csv(OUT['gap_disposition'],index=False)
uid_manifest.to_csv(OUT['uid_manifest'],index=False)
core_fields.to_csv(OUT['core_audit'],index=False)
core_requirements.to_csv(OUT['core_requirements'],index=False)
# Do not overwrite manually reviewed template on re-run. Save a fresh starter only if absent.
if not OUT['bindings'].exists(): bindings.to_csv(OUT['bindings'],index=False)
if patched_bundle is not None:
    with OUT['patched_fhir'].open('w',encoding='utf-8') as f: json.dump(patched_bundle,f,indent=2)

summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'28_official_core_contract_and_synthetic_required_field_disposition',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Synthetic-only; no real ADNI records, real DICOM identifiers or clinical source events exported.',
 'phase27_repaired_sha256':filehash(P27_PATH),
 'phase27_fields_sha256':filehash(INP['fields']),
 'phase27_gaps_sha256':filehash(INP['gaps']),
 'phase26_original_bundle_sha256':expected_bundle_hash,
 'imagewg':{
  'commit':IMAGEWG_COMMIT,'pinned_field_source_sha256':P27['pinned_field_level_source_sha256'],
  'phase27_schema_previously_verified':True,'network_recheck_passed':imagewg_rechecked,
  'network_recheck_error':imagewg_error
 },
 'omop_cdm_v55':{
  'verified':core_verified,'pinned_commit':CORE_COMMIT if core_verified else None,
  'field_csv_sha256':core_sha if core_verified else None,
  'official_core_schema_error':core_error
 },
 'synthetic_uid':{
  'phase26_bundle_verified':bundle is not None,
  'image_occurrence_staging_row_reconciled':bool(uid_row_gate),
  'synthetic_study_and_series_pairs_generated':len(uid_rows),
  'synthetically_resolved_uid_required_fields':resolved,
  'real_dicom_uid_semantics_attested':False,
  'patched_bundle_sha256':filehash(OUT['patched_fhir']) if patched_bundle is not None else None,
 },
 'gap_disposition':{
  'original_required_micdm_instances':48,
  'synthetic_only_resolved':resolved,
  'still_blocked':remaining,
  'blocked_by_field':[{'table':str(t),'field':str(f),'count':int(n)} for (t,f),n in disposition[disposition['resolution_status'].str.startswith('BLOCKED')].groupby(['table','required_field']).size().items()]
 },
 'gates':{
  'phase27_repaired_verified':True,
  'exact_imagewg_fields_verified_in_phase27':True,
  'imagewg_network_recheck_passed':bool(imagewg_rechecked),
  'omop_v55_core_fields_verified_from_commit':bool(core_verified),
  'synthetic_uid_demo_generated':bool(len(uid_rows)==4),
  'synthetic_uid_staging_gap_closure_verified':bool(uid_row_gate),
  'Athena_mappings_approved':False,
  'source_linked_procedure_attested':False,
  'real_DICOM_UIDs_attested':False,
  'image_feature_event_field_concept_validated':False,
  'synthetic_faithful_armA_mapping_completed':False,
  'pyOMOP_ETL_executed':False,
  'official_MICDM_load_completed':False,
  'real_MRI_rows_processed':0,
  'true_four_arm_comparison_completed':False,
  'OMOP_target_cohort_equivalence_tested':False
 },
 'arm_registry':{
  'A':'SYNTHETIC_PREPARATION_ONLY_REQUIRED_BINDINGS_UNRESOLVED',
  'B':'NOT_EXECUTED',
  'C':'ORIGINAL_GENERIC_TFL_NONPORTABLE_NOT_EXECUTED',
  'D':'SYNTHETIC_CONTRACT_PROTOTYPE_ONLY'
 },
 'next_stage_gate':(
  'Review actual source/procedure provenance or add explicitly labeled synthetic Procedure test fixtures; '
  'validate core PERSON/PROCEDURE_OCCURRENCE/MEASUREMENT fields and Athena concepts using a frozen local vocabulary; '
  'bind valid IMAGE_FEATURE event-field concept; only then test faithful synthetic Arm A target cardinality, idempotence and cohorts. '
  'Never replace real imaging UIDs with synthetic test values.'
 )
}
with OUT['shareable'].open('w',encoding='utf-8') as f:json.dump(summary,f,indent=2)
for name,path in OUT.items():
    if name=='patched_fhir' and patched_bundle is None:continue
    if not path.is_file():raise RuntimeError('Output was not written: '+str(path))
    assert path.stat().st_size>0
with OUT['shareable'].open(encoding='utf-8') as f: final=json.load(f)
assert final['gap_disposition']['original_required_micdm_instances']==48
assert final['gap_disposition']['synthetic_only_resolved']+final['gap_disposition']['still_blocked']==48
assert final['gates']['synthetic_faithful_armA_mapping_completed'] is False
assert final['gates']['real_DICOM_UIDs_attested'] is False
assert final['gates']['OMOP_target_cohort_equivalence_tested'] is False
assert len(pd.read_csv(OUT['gap_disposition']))==48
print('PHASE 28 PREPARATION COMPLETE: all output files written to',WORKDIR)
for name,path in OUT.items():
    if path.exists(): print(' -',path.name)
print('Required gaps: synthetically resolved =',resolved,', still blocked =',remaining)
print('Independent core schema verified:',core_verified, '| Full Arm A mapping: NOT EXECUTED')

PHASE 28 PREPARATION COMPLETE: all output files written to /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
 - ICHI2027_Phase28_SHAREABLE.json
 - ICHI2027_Phase28_GAP_DISPOSITION.csv
 - ICHI2027_Phase28_SYNTHETIC_UID_MANIFEST.csv
 - ICHI2027_Phase28_PINNED_OMOP55_CORE_FIELDS.csv
 - ICHI2027_Phase28_CORE_REQUIREMENTS.csv
 - ICHI2027_Phase28_CONCEPT_AND_PROCEDURE_BINDING_TEMPLATE.csv
 - ICHI2027_Phase28_SYNTHETIC_FHIR_UID_PATCHED.json
Required gaps: synthetically resolved = 8 , still blocked = 40
Independent core schema verified: True | Full Arm A mapping: NOT EXECUTED
